[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/05_numerical_stability_in_deep_learning/exercises.ipynb)

# Exercises — Topic 05: Numerical Stability in Deep Learning

20 fully solved problems in 4 levels. Attempt each before opening its solution cell.

Constants used throughout: fp16 — $u = 2^{-11} = 4.9 \times 10^{-4}$, min normal $6.1 \times 10^{-5}$, min subnormal $2^{-24} = 6.0 \times 10^{-8}$, max $65504$. bf16 — $u = 2^{-8} = 3.9 \times 10^{-3}$, range as fp32. fp32 — $u = 2^{-24} = 6.0 \times 10^{-8}$, min subnormal $1.4 \times 10^{-45}$, max $3.4 \times 10^{38}$. fp8 E4M3 — $u = 2^{-4}$, min normal $2^{-9} = 2.0 \times 10^{-3}$, max $448$.

## Level 0 — Concept Check

### Problem L0.1: How large a logit can each format exponentiate?

Compute the largest $z$ for which $e^{z}$ is representable in fp64, fp32, bf16, fp16, and fp8 E4M3. Then state whether a transformer with $d_k = 64$ and unnormalized attention scores is safe in fp16.

**Solution**

The threshold is $z_{\max} = \log X_{\max}$:

| Format | $X_{\max}$ | $z_{\max} = \log X_{\max}$ |
|---|---|---|
| fp64 | $1.8 \times 10^{308}$ | $709.78$ |
| fp32 | $3.4 \times 10^{38}$ | $88.72$ |
| bf16 | $3.4 \times 10^{38}$ | $88.72$ |
| fp16 | $65504$ | $11.09$ |
| fp8 E4M3 | $448$ | $6.10$ |

**Attention.** With $q, k \in \mathbb{R}^{64}$ having i.i.d. unit-variance entries, $q^{\top}k$ has mean $0$ and standard deviation $\sqrt{d_k} = 8$. A $2\sigma$ score is $16 \gt 11.09$, so **a substantial fraction of unnormalized scores overflow in fp16** — not an edge case but routine operation. Dividing by $\sqrt{d_k}$ returns the scores to unit scale; combined with max-subtraction the softmax is then unconditionally safe.

$$
\boxed{z_{\max}: \; 709.8 \; (\text{fp64}), \; 88.7 \; (\text{fp32/bf16}), \; 11.09 \; (\text{fp16}), \; 6.10 \; (\text{fp8})}
$$

*Key takeaway*: $11.09$ is the number to remember. Logits, attention scores, and energies are all $O(10)$ quantities, so fp16 exponentials require the max-subtraction *and* a scale factor.

### Problem L0.2: bf16 or fp16?

Both are 16 bits. (a) State the bit split and the resulting range/precision of each. (b) Which needs loss scaling, and why? (c) Which computes $\sum_{i=1}^{4096} x_i$ more accurately? (d) Which would you pick for gradient all-reduce?

**Solution**

**(a)** fp16 is 1/5/10 (sign/exponent/mantissa): $p = 11$, $u = 4.9\times10^{-4}$, range $[6.1\times10^{-5}, 65504]$. bf16 is 1/8/7: $p = 8$, $u = 3.9\times10^{-3}$, range $[1.2\times10^{-38}, 3.4\times10^{38}]$ — identical to fp32.

**(b)** **fp16** needs loss scaling. Its lower range limit ($2^{-24}$ including subnormals) is far above the small-gradient tail, so components underflow to zero. bf16 shares fp32's exponent field: any value representable in fp32 survives the cast, so no scaling is required. This is the reason bf16 has displaced fp16 in large-scale pretraining — one fewer failure mode at a scale where a divergence is expensive.

**(c)** **fp16**, by $8\times$: with $u_{16} = 4.9\times10^{-4}$ versus $u_{\text{bf}} = 3.9\times10^{-3}$, the recursive-summation bound $\gamma_{n} \approx nu$ gives $2.0$ versus $16$ for $n = 4096$ — both useless, which is the real lesson: **neither format may accumulate a 4096-term sum.** Use an fp32 accumulator (bound $2.4\times10^{-4}$).

**(d)** **bf16**, and accumulate in fp32. Gradients have a wide dynamic range across layers and steps, so range beats precision; the reduction is bandwidth-bound (Topic 04), so 16-bit transport is the point; and tree topology keeps the error depth at $\log_2 W$ rather than $W$ (Topic 02).

$$
\boxed{\text{fp16} = \text{precision in a narrow window}; \; \text{bf16} = \text{fp32's window at } 8\times \text{ coarser} }
$$

*Key takeaway*: 16 bits is a budget, and the two formats spend it on opposite failure modes. Choose by asking whether your risk is range or precision.

### Problem L0.3: Loss scaling — true or false

(a) "Loss scaling changes the loss landscape, so it must be tuned like a learning rate."
(b) "Loss scaling introduces extra rounding error."
(c) "Loss scaling can fix gradients that overflow."
(d) "bf16 training should also use loss scaling."


**Solution**

**(a) False.** By linearity, $\nabla(S\mathcal{L}) = S\nabla\mathcal{L}$; dividing the gradient by $S$ before the update recovers the identical parameter trajectory. It is a change of *units* for the gradient, invisible to the optimizer. (It does interact with anything that reads raw gradient magnitudes — clipping thresholds, weight decay applied to gradients — so unscale *before* those, which is exactly what framework `GradScaler.unscale_()` does.)

**(b) False.** Practical implementations use $S = 2^{k}$, and multiplying or dividing by a power of two only modifies the exponent field: **exact**, no rounding at all (barring overflow/underflow, which is the point of choosing $S$ correctly).

**(c) False, backwards.** Scaling multiplies gradients *up*; it cures **underflow**. Overflow is cured by *decreasing* $S$ — which is precisely what dynamic loss scaling does when it detects a non-finite gradient.

**(d) False.** bf16 has fp32's exponent range, so nothing that was representable in fp32 underflows. Loss scaling would add risk (overflow at the top) with no benefit.

$$
\boxed{\text{(a) F, (b) F, (c) F, (d) F — loss scaling is an exact, range-only, fp16-only device}}
$$

*Key takeaway*: loss scaling is the cheapest intervention in the module — mathematically a no-op, numerically decisive, and free of rounding cost.

### Problem L0.4: Where does $\epsilon$ go?

(a) In LayerNorm, why is $\epsilon$ inside the square root: $(\sigma^{2} + \epsilon)^{-1/2}$ rather than $(\sigma + \epsilon)^{-1}$? (b) In Adam, what does $\epsilon$ actually control, and what happens if it is stored in fp16 with the default value $10^{-8}$?

**Solution**

**(a)** Two reasons, both about the degenerate case $\sigma \to 0$ (a constant feature, a batch of size 1, a masked-out channel):

- *Forward*: both placements keep the output finite, but $(\sigma^{2}+\epsilon)^{-1/2}$ bounds it by $\vert x - \mu \vert/\sqrt{\epsilon}$ with a smooth dependence on $\sigma^{2}$, which is the quantity actually computed.
- *Backward*: $\frac{\partial}{\partial\sigma^{2}}(\sigma^{2}+\epsilon)^{-1/2} = -\tfrac{1}{2}(\sigma^{2}+\epsilon)^{-3/2}$ is bounded by $\tfrac{1}{2}\epsilon^{-3/2}$. With $\epsilon$ outside, the chain must pass through $\sigma = \sqrt{\sigma^{2}}$, whose derivative $\frac{1}{2\sigma}$ is **unbounded** as $\sigma \to 0$: a `NaN` gradient on a constant batch.

**(b)** $\epsilon$ caps the condition number of Adam's diagonal preconditioner:

$$
\kappa(P) \le \frac{\sqrt{v_{\max}} + \epsilon}{\epsilon}
$$

and hence bounds the maximum step at $\eta\hat{m}/\epsilon$. It is Tikhonov regularization (Topic 03, Derivation 3.6), not a division guard. Stored in fp16, $10^{-8}$ is below the smallest subnormal $6.0 \times 10^{-8}$ and **flushes to exactly zero**, restoring the unbounded step for any coordinate whose $\hat{v}$ has decayed — a classic source of mid-training `inf`. Hence optimizer state lives in fp32.

$$
\boxed{\text{LayerNorm: inside the root, for a bounded } \tfrac{\partial}{\partial\sigma^{2}}; \quad \text{Adam: } \epsilon \text{ caps } \kappa(P), \text{ and } 10^{-8} \to 0 \text{ in fp16}}
$$

*Key takeaway*: every $\epsilon$ in deep learning is a condition-number cap. Its value and its position are both load-bearing.

## Level 1 — Foundation

### Problem L1.1: Softmax by hand, both ways

Logits $z = (12, 10, 11)$, arithmetic fp16. (a) What does the naive softmax return? (b) Compute the stable version by hand. (c) Compute the cross-entropy loss for true class $y = 1$ (the logit $10$) using the fused formula.

**Solution**

**(a) Naive.** $e^{12} = 162754.8 \gt 65504$, so $\mathrm{fl}_{16}(e^{12}) = \infty$. Likewise $e^{11} = 59874 \lt 65504$ survives and $e^{10} = 22026$ survives. The sum is $\infty$, and

$$
\mathrm{softmax}_0 = \frac{\infty}{\infty} = \mathrm{NaN}
$$

Every output is `NaN` — from logits that are entirely ordinary.

**(b) Stable.** $m = \max_j z_j = 12$; shifted logits $(0, -2, -1)$:

$$
e^{0} = 1, \quad e^{-2} = 0.13534, \quad e^{-1} = 0.36788, \qquad \Sigma = 1.50322
$$

$$
\mathrm{softmax}(z) = \left( \frac{1}{1.50322}, \frac{0.13534}{1.50322}, \frac{0.36788}{1.50322} \right) = (0.66524, \; 0.09003, \; 0.24473)
$$

All intermediates lie in $[0.135, 1.51]$ — comfortably inside fp16, indeed inside fp8.

**(c) Fused loss** for $y = 1$ ($z_y = 10$):

$$
\ell = -z_y + m + \log\Sigma = -10 + 12 + \log(1.50322) = 2 + 0.40761 = 2.40761
$$

Check: $-\log(0.09003) = 2.4076$ ✓ — but computed without ever forming $0.09003$.

$$
\boxed{\text{naive} \to \mathrm{NaN}; \quad p = (0.6652, 0.0900, 0.2447); \quad \ell = 2.4076}
$$

*Key takeaway*: the failure needs no adversarial input — three two-digit logits suffice. Max-subtraction is not a hardening measure, it is the algorithm.

### Problem L1.2: Derive `BCEWithLogitsLoss`

Starting from $\ell = -[y\log\sigma(z) + (1-y)\log(1-\sigma(z))]$ with $\sigma(z) = (1+e^{-z})^{-1}$: (a) derive the stable form; (b) evaluate at $z = -30$, $y = 1$ in fp32, both ways; (c) give the gradient and note its range.

**Solution**

**(a)** Use $\log\sigma(z) = -\log(1+e^{-z})$ and $\log(1-\sigma(z)) = \log\frac{e^{-z}}{1+e^{-z}} = -z - \log(1+e^{-z})$:

$$
\ell = y\log(1+e^{-z}) + (1-y)\left[ z + \log(1+e^{-z}) \right] = (1-y)z + \log(1 + e^{-z})
$$

This still overflows $e^{-z}$ for $z \ll 0$. Factor $e^{-\max(z,0)}$ out of $1 + e^{-z}$, i.e. write $\log(1+e^{-z}) = \max(-z, 0) + \log(1 + e^{-\vert z \vert})$, giving

$$
\boxed{\ell = \max(z, 0) - yz + \mathrm{log1p}\!\left( e^{-\vert z \vert} \right)}
$$

**(b) At $z = -30$, $y = 1$.**

- *Naive*: $\sigma(-30) = 9.36\times10^{-14}$, computed as $1/(1+e^{30}) = 1/(1 + 1.07\times10^{13})$ — representable in fp32, so $\log\sigma = -30.0000$. It happens to survive here, but at $z = -104$ the term $e^{-z} = e^{104}$ overflows fp32 and returns `inf`, making the loss `inf`. In fp16 it fails already at $z = -12$.
- *Stable*: $\max(-30, 0) = 0$, $-yz = 30$, $\mathrm{log1p}(e^{-30}) = \mathrm{log1p}(9.36\times10^{-14}) = 9.36\times10^{-14}$:

$$
\ell = 0 + 30 + 9.36\times10^{-14} = 30.0000000000001
$$

correct to full precision, and correct for every $z$ down to the format's limits.

**(c) Gradient.**

$$
\frac{\partial\ell}{\partial z} = \sigma(z) - y \; \in \; [-1, 1]
$$

Bounded regardless of how wrong the prediction is — the same structural property as softmax cross-entropy, and the reason the fused kernel is safe in the backward pass where a naive $\frac{1}{\sigma}\frac{\partial\sigma}{\partial z}$ would overflow.

*Key takeaway*: three algebraic steps convert an `inf`-producing expression into one accurate over the entire real line, at lower cost.

### Problem L1.3: Choosing a loss scale

A gradient histogram (fp32 reference) has $g_{\max} = 2^{-4}$ and a lower tail down to $g_{\min} = 2^{-30}$, to be represented in fp16. (a) What fraction of the histogram is lost without scaling? (b) Derive the feasible range of $S$. (c) Pick $S$ and justify. (d) What is the largest dynamic range fp16 can accommodate at all?

**Solution**

**(a)** fp16 rounds to zero below $2^{-25}$ (half the smallest subnormal $2^{-24}$). The tail from $2^{-30}$ to $2^{-25}$ — five binades of the histogram — vanishes, and the corresponding weights receive no update. Values between $2^{-24}$ and $2^{-14}$ survive only as subnormals with degraded precision (and become $0$ on hardware with flush-to-zero enabled).

**(b)** Two-sided constraint:

$$
S\,g_{\min} \gt 2^{-24} \Rightarrow S \gt \frac{2^{-24}}{2^{-30}} = 2^{6} = 64
$$

$$
S\,g_{\max} \lt 65504 \Rightarrow S \lt \frac{65504}{2^{-4}} = 65504 \times 16 \approx 2^{20}
$$

So $S \in (2^{6}, 2^{20})$.

**(c)** Choose the geometric centre of the feasible interval, $S = 2^{13} = 8192$: it leaves $7$ binades of margin at each end, absorbing histogram drift in both directions during training. The conventional starting value $S = 2^{16}$ is also feasible here but leaves only $4$ binades before overflow — acceptable because dynamic loss scaling will back off automatically on the first non-finite gradient.

**(d)** Feasibility requires $\frac{2^{-24}}{g_{\min}} \lt \frac{65504}{g_{\max}}$, i.e.

$$
\frac{g_{\max}}{g_{\min}} \lt 65504 \times 2^{24} \approx 1.1 \times 10^{12} \approx 2^{40}
$$

Any gradient histogram spanning more than $40$ binades cannot fit in fp16 at any single scale — the case that motivates per-tensor scaling (fp8) or bf16.

$$
\boxed{S \in (2^{6}, 2^{20}); \; \text{choose } S = 2^{13}; \; \text{fp16 admits a dynamic range up to } 2^{40}}
$$

*Key takeaway*: pick $S$ at the geometric centre of the feasible window, then let dynamic scaling track the drift.

### Problem L1.4: When does a weight update disappear?

A weight $\theta = 0.1$ receives an update $\eta g$ with $\eta = 10^{-3}$, $g = 10^{-3}$. (a) Compute the relative update. (b) Determine, for fp16, bf16, and fp32 storage, whether the update survives. (c) Explain why the fp32 master copy is not merely conservative.

**Solution**

**(a)**

$$
\frac{\eta\vert g \vert}{\vert\theta\vert} = \frac{10^{-3}\times10^{-3}}{0.1} = 10^{-5}
$$

**(b)** The update is absorbed when the relative update is below $u/2$ (it then rounds back to $\theta$):

| Format | $u/2$ | $10^{-5}$ vs $u/2$ | Outcome |
|---|---|---|---|
| fp16 | $2.4\times10^{-4}$ | $10^{-5} \lt 2.4\times10^{-4}$ | **absorbed — no change** |
| bf16 | $2.0\times10^{-3}$ | $10^{-5} \lt 2.0\times10^{-3}$ | **absorbed — no change** |
| fp32 | $6.0\times10^{-8}$ | $10^{-5} \gt 6.0\times10^{-8}$ | survives, $\approx 166$ ulps |

**(c)** Absorption is not a small loss of accuracy — it is a **complete stop**. In 16-bit storage the weight is frozen for as long as the relative update stays below $u/2$, which for typical late-stage training is *always*. Training appears to proceed (the loss still moves because other weights with larger relative updates still change) but a growing fraction of parameters is inert, and the run silently converges to the wrong place.

The fp32 master copy fixes this by letting sub-ulp updates accumulate: after $\approx 24$ steps of $10^{-5}$ relative each, the accumulated change crosses fp16's grid and the next cast moves the compute copy. This is precisely Topic 02's "track residuals, not absolutes" — the same structure as Kahan summation, where a compensation variable preserves what the addition dropped.

$$
\boxed{\text{relative update } 10^{-5} \text{ is absorbed by fp16 } (u/2 = 2.4\times10^{-4}) \text{ and bf16, retained by fp32}}
$$

*Key takeaway*: compare $\frac{\eta\Vert g \Vert}{\Vert\theta\Vert}$ against $u/2$ before choosing a weight dtype. It is a one-line check that predicts a class of silent failures.

### Problem L1.5: How long a reduction can each format sustain?

Using the recursive-summation bound $\gamma_n \approx nu$: (a) find the length $n$ at which the relative error bound reaches $100\%$ for fp16, bf16, fp32, fp64. (b) Apply to a dot product with $H = 4096$. (c) What do tensor cores do, and what does pairwise summation buy?

**Solution**

**(a)** $\gamma_n \approx nu = 1 \Rightarrow n = 1/u$:

| Format | $u$ | $n_{\text{stall}} = 1/u$ |
|---|---|---|
| fp16 | $4.9\times10^{-4}$ | $2048$ |
| bf16 | $3.9\times10^{-3}$ | $256$ |
| fp32 | $6.0\times10^{-8}$ | $1.7\times10^{7}$ |
| fp64 | $1.1\times10^{-16}$ | $9\times10^{15}$ |

(Equivalently $n_{\text{stall}} = 2^{p}$: the point at which a new addend of comparable size falls below the accumulator's ulp.)

**(b) $H = 4096$.** In fp16: $\gamma_{4096} = 4096 \times 4.9\times10^{-4} = 2.0$ — a bound of $200\%$, i.e. the result carries no guaranteed digits. In bf16: $\gamma_{4096} = 16$, worse still. In fp32: $\gamma_{4096} = 2.4\times10^{-4}$ — about $3.6$ correct digits, adequate.

**(c)** Tensor cores read 16-bit (or 8-bit) inputs and **accumulate in fp32 by hardware design** — the multiply is narrow, the addition tree is wide — which is exactly why mixed precision works at all. Additionally, hardware and library reductions are *tree-shaped*, replacing $\gamma_n \approx nu$ with $\gamma_{\log_2 n} \approx u\log_2 n$: for $n = 4096$ that is $12u$ instead of $4096u$, a further $340\times$. Combining both, a $4096$-long bf16 dot product with fp32 tree accumulation has error bound $\approx 12 \times 6\times10^{-8} = 7\times10^{-7}$ from accumulation, dominated instead by the $u_{\text{bf}} = 3.9\times10^{-3}$ input rounding — the correct design point, since input precision is what the format was chosen for.

$$
\boxed{n_{\text{stall}} = 1/u: \; 2048 \; (\text{fp16}), \; 256 \; (\text{bf16}); \; \text{hence fp32 tree accumulation is mandatory}}
$$

*Key takeaway*: the reduction length, compared against $1/u$, decides the accumulator width. For any hidden size above a few hundred, 16-bit accumulation is arithmetically impossible.

### Problem L1.6: Variance of activations in low precision

A channel has activations with mean $\mu = 20$ and standard deviation $\sigma = 0.5$. (a) Compute the cancellation amplification of the one-pass variance formula. (b) Give the resulting relative error in bf16 and fp32. (c) State the two fixes and why RMSNorm sidesteps the issue.

**Solution**

**(a)** The one-pass identity $\sigma^{2} = \overline{x^{2}} - \mu^{2}$ subtracts two quantities of size $\approx \mu^{2} = 400$ to obtain $\sigma^{2} = 0.25$. Amplification (Topic 02, Theorem 2.6):

$$
\frac{\overline{x^{2}} + \mu^{2}}{\sigma^{2}} \approx \frac{2\mu^{2}}{\sigma^{2}} = \frac{800}{0.25} = 3200
$$

**(b)** Multiply by the operands' relative error, at least $u$:

- **bf16** ($u = 3.9\times10^{-3}$): relative error $\ge 3200 \times 3.9\times10^{-3} = 12.5$, i.e. **$1250\%$** — the computed variance is noise and can be negative, making $\sqrt{\sigma^{2}}$ a `NaN`.
- **fp32** ($u = 6\times10^{-8}$): $3200 \times 6\times10^{-8} = 1.9\times10^{-4}$, about $0.02\%$ — acceptable, and this is before the summation error over the batch, which adds a further factor of $\gamma_n$.

**(c) Fixes.**
1. **Compute statistics in fp32** even when activations are 16-bit (every framework does this).
2. **Use the two-pass or Welford recurrence**, which subtracts only residuals $x_i - m$ of size $\sigma$: the amplification factor becomes $1$ and the error stays $O(u)$ relative to $\sigma^{2}$ itself — a $3200\times$ improvement here.

**RMSNorm** normalizes by $\sqrt{\overline{x^{2}} + \epsilon}$ without subtracting the mean. There is no subtraction of large nearly equal quantities at all, so the cancellation cannot occur — one of the underappreciated reasons it is both cheaper and more robust than LayerNorm at scale (though it does change the function being computed).

$$
\boxed{\text{amplification } 2\mu^{2}/\sigma^{2} = 3200: \text{ bf16 gives } 1250\% \text{ error, fp32 gives } 0.02\%; \text{ Welford removes it}}
$$

*Key takeaway*: compute $\mu^{2}/\sigma^{2}$ for your activations before trusting any variance. It predicts, in advance, which precision the statistics path requires.

## Level 2 — Applications in AI/ML

### Problem L2.1: The $1/\sqrt{d_k}$ in attention is a numerical constant

With $q, k \in \mathbb{R}^{d}$ having i.i.d. zero-mean unit-variance entries and $d = 128$: (a) compute the mean and standard deviation of $q^{\top}k$; (b) estimate the maximum over $N = 4096$ keys with and without the $1/\sqrt{d}$ factor; (c) relate to fp16's limit and to the softmax Jacobian.

**Solution**

**(a)** $\mathbb{E}[q^{\top}k] = \sum_i \mathbb{E}[q_i]\mathbb{E}[k_i] = 0$, and by independence

$$
\mathrm{Var}(q^{\top}k) = \sum_{i=1}^{d}\mathrm{Var}(q_i k_i) = d \quad \Longrightarrow \quad \mathrm{sd} = \sqrt{d} = 11.3
$$

**(b)** The maximum of $N$ approximately-Gaussian scores is $\approx \mathrm{sd}\sqrt{2\log N}$:

$$
\sqrt{2\log 4096} = \sqrt{2 \times 8.32} = 4.08
$$

- **Unscaled**: $\max \approx 11.3 \times 4.08 = 46.2$. Then $e^{46.2} = 1.2\times10^{20}$ — overflows fp16 ($z_{\max} = 11.09$) by nine orders of magnitude, and overflows fp8 catastrophically.
- **Scaled by $1/\sqrt{d}$**: scores have unit standard deviation, $\max \approx 4.08$, and $e^{4.08} = 59$ — comfortably inside every format, even before max-subtraction.

**(c)** Two consequences of the scaling, both essential:

1. **Range**: it keeps the pre-softmax values inside the exponent window. Combined with max-subtraction, overflow becomes structurally impossible.
2. **Conditioning**: the softmax Jacobian is $J = \frac{1}{T}\left( \mathrm{diag}(p) - pp^{\top} \right)$ with $T$ the effective temperature. Unscaled scores are equivalent to $T = 1/\sqrt{d}$, i.e. a $\sqrt{d} = 11.3$-fold sharper softmax: the distribution saturates toward a one-hot, the Jacobian's nonzero entries collapse toward zero, and gradients vanish. The original Transformer paper motivates $1/\sqrt{d_k}$ by exactly this gradient argument; the overflow argument is the numerical twin.

$$
\boxed{\mathrm{sd}(q^{\top}k) = \sqrt{d} = 11.3, \; \max \approx 46 \text{ unscaled} \gg 11.09; \; \text{scaled: } \max \approx 4.1}
$$

*Key takeaway*: a constant that looks like a modelling choice ($1/\sqrt{d_k}$) is doing double duty as a range guard and a conditioning fix — the recurring pattern of this module.

### Problem L2.2: Trace dynamic loss scaling

A run starts with $S = 2^{16}$, growth interval $2000$ steps, halving on overflow. Overflows occur at steps $150$, $151$, and $4300$. (a) Trace $S$ through step $6500$. (b) How many updates are lost? (c) Why is skipping the step correct rather than clipping the gradient?

**Solution**

**(a) Trace.**

| Step | Event | $S$ after | Note |
|---|---|---|---|
| 0 | start | $2^{16} = 65536$ | — |
| 150 | overflow | $2^{15} = 32768$ | step skipped, success counter reset |
| 151 | overflow | $2^{14} = 16384$ | step skipped, counter reset |
| 2151 | 2000 successes | $2^{15} = 32768$ | growth |
| 4151 | 2000 successes | $2^{16} = 65536$ | growth |
| 4300 | overflow | $2^{15} = 32768$ | step skipped, counter reset |
| 6300 | 2000 successes | $2^{16} = 65536$ | growth |
| 6500 | — | $2^{16}$ | steady |

**(b) Lost updates**: exactly $3$ (steps $150$, $151$, $4300$) out of $6500$ — $0.046\%$ of training. Statistically invisible; the optimizer's momentum carries through the gap.

**(c)** Because the gradient is **not merely large — it is wrong**. An overflow produces `inf`, and `inf` in a gradient carries no directional information at all; clipping it (`inf` $\to c$) would fabricate a direction, and `inf - inf` or `0 * inf` elsewhere in the reduction produces `NaN`, which would then poison the Adam moment buffers permanently ($\mathrm{NaN}\cdot\beta + \dots = \mathrm{NaN}$ forever). Skipping discards a corrupted measurement, which is the statistically and numerically correct action, and simultaneously signals that $S$ is too large.

Note the asymmetry that makes the algorithm work: overflow is **detectable** (a finiteness check on the gradient, one cheap reduction), while underflow is **silent** (a zero looks like a legitimate zero). So the controller drives $S$ upward blindly and backs off on the one observable signal.

$$
\boxed{S: 2^{16} \to 2^{14} \text{ (steps 150–151)} \to 2^{16} \text{ (step 4151)} \to 2^{15} \to 2^{16}; \; 3 \text{ steps lost } (0.05\%)}
$$

*Key takeaway*: dynamic loss scaling is a feedback controller whose only sensor is an overflow flag — it maximizes $S$ subject to the one constraint it can measure.

### Problem L2.3: The memory bill of mixed precision

For a $7$B-parameter model trained with bf16 compute, fp32 master weights, and Adam: (a) compute bytes per parameter for each stored tensor and the total; (b) the total in GB; (c) compare with pure fp32 training; (d) what do ZeRO sharding and 8-bit optimizers change?

**Solution**

**(a) Per parameter.**

| Tensor | Format | Bytes |
|---|---|---|
| Compute weights | bf16 | 2 |
| Gradients | bf16 | 2 |
| Master weights | fp32 | 4 |
| Adam first moment $m$ | fp32 | 4 |
| Adam second moment $v$ | fp32 | 4 |
| **Total** | — | **16** |

**(b)** $7\times10^{9} \times 16 = 1.12\times10^{11}$ bytes $= 112$ GB — before any activations, which add $O(\text{batch} \times \text{seq} \times \text{layers} \times H)$ more. A single 80 GB A100 cannot hold even the model state.

**(c) Pure fp32**: weights 4 + gradients 4 + $m$ 4 + $v$ 4 $= 16$ bytes — **identical**. The striking conclusion: mixed precision does **not** save model-state memory. Its wins are (i) *activation* memory (halved, and activations dominate at large batch/sequence), and (ii) *throughput*, via tensor cores and halved bandwidth (Topic 04).

**(d)**
- **ZeRO / FSDP** shards the optimizer state, gradients, and parameters across $W$ data-parallel ranks: per-rank cost falls to $\approx 16/W$ bytes per parameter (stage 3), at the price of extra all-gather communication — the trade that made 100B+ models trainable.
- **8-bit optimizers** (block-wise dynamic quantization of $m$ and $v$) cut the two moment tensors from 8 bytes to 2, taking the total from 16 to 10 bytes per parameter — a $37\%$ saving. They work because the moments are *statistics*, not parameters: their quantization error is averaged out over steps, whereas quantizing the master weights would reintroduce the absorption problem of Problem L1.4.

$$
\boxed{16 \text{ bytes/param} = 112\,\text{GB for 7B; identical to fp32 — the win is activations and throughput, not state}}
$$

*Key takeaway*: the fp32 master copy and optimizer state, not the weights, dominate memory. This is why sharding and optimizer quantization — not weight precision — are the levers for fitting large models.

### Problem L2.4: Porting Adam between frameworks

Framework X uses $\frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon_X}$ with $\epsilon_X = 10^{-8}$; framework Y uses $\frac{\hat{m}}{\sqrt{\hat{v} + \epsilon_Y}}$. (a) Find the $\epsilon_Y$ that matches X's behaviour for small $\hat{v}$. (b) What goes wrong if a user copies $10^{-8}$ across? (c) What is the maximum step in each case? (d) What if $\hat{v}$ is stored in fp16?

**Solution**

**(a)** For $\hat{v} \to 0$, X's denominator tends to $\epsilon_X$ and Y's to $\sqrt{\epsilon_Y}$. Matching:

$$
\sqrt{\epsilon_Y} = \epsilon_X \quad \Longrightarrow \quad \epsilon_Y = \epsilon_X^{2} = 10^{-16}
$$

**(b)** Copying $\epsilon_Y = 10^{-8}$ gives an effective floor $\sqrt{10^{-8}} = 10^{-4}$ — **four orders of magnitude larger** than intended. Coordinates with $\sqrt{\hat{v}} \lt 10^{-4}$ (i.e. any consistently small gradient) are damped by a factor of up to $10^{4}$, and Adam degenerates toward SGD-with-momentum at learning rate $\eta/10^{-4}$ for those coordinates. Symptoms: a run that trains but plateaus early, or one whose effective learning rate is silently wrong by orders of magnitude — a real and frequently reported porting bug.

**(c) Maximum step magnitudes** (taking $\hat{m}$ at its bound):

$$
\text{X}: \; \left\vert \Delta\theta \right\vert \le \frac{\eta\vert\hat{m}\vert}{\epsilon_X} = 10^{8}\eta\vert\hat{m}\vert, \qquad \text{Y}: \; \left\vert \Delta\theta \right\vert \le \frac{\eta\vert\hat{m}\vert}{\sqrt{\epsilon_Y}}
$$

In practice both are far above the typical step $\approx \eta$ (attained when $\vert\hat{m}\vert \approx \sqrt{\hat{v}}$), which is why Adam's step size is often described as "$\approx \eta$ regardless of gradient scale" — but the cap matters exactly in the small-gradient regime where the preconditioner would otherwise be singular.

**(d)** In fp16: $\epsilon_X = 10^{-8}$ is below the smallest subnormal $6\times10^{-8}$ and **flushes to zero**, removing the cap entirely; and $\epsilon_Y = 10^{-16}$ is even further gone. Worse, $\hat{v} \approx g^{2}$ for $g \sim 10^{-4}$ is $10^{-8}$ — itself below fp16's subnormal floor, so $\hat{v}$ underflows to zero and the update becomes $0/0$. Both facts force Adam's state into fp32 (or into 8-bit *with a per-block scale*, which restores the range).

$$
\boxed{\epsilon_Y = \epsilon_X^{2} = 10^{-16}; \text{ copying } 10^{-8} \text{ raises the floor } 10^{4}\times; \text{ both underflow in fp16}}
$$

*Key takeaway*: $\epsilon$ is not a portable constant. Convert it through the placement, and keep it — and $\hat{v}$ — in a format whose range accommodates $g^{2}$.

### Problem L2.5: How deep can each format go?

Assume per-layer gradient contraction $\mathbb{E}[\log\sigma] = \log 0.9 = -0.1054$ and an output gradient of order $1$. (a) Derive the depth $L^{*}$ at which the gradient underflows for fp16 and fp32. (b) Recompute for fp16 with loss scaling $S = 2^{16}$. (c) What do residual connections do to this analysis?

**Solution**

**(a)** The gradient norm at depth $0$ is $\approx e^{L\,\mathbb{E}[\log\sigma]}$; underflow occurs when this falls below the smallest representable positive value $x_{\min}$:

$$
L\,\mathbb{E}[\log\sigma] \lt \log x_{\min} \quad \Longrightarrow \quad L^{*} = \frac{\log x_{\min}}{\mathbb{E}[\log\sigma]}
$$

- **fp16**: $x_{\min} = 6.0\times10^{-8}$, $\log x_{\min} = -16.63$:

$$
L^{*} = \frac{-16.63}{-0.1054} \approx 158 \text{ layers}
$$

- **fp32**: $x_{\min} = 1.4\times10^{-45}$, $\log x_{\min} = -103.3$:

$$
L^{*} = \frac{-103.3}{-0.1054} \approx 980 \text{ layers}
$$

**(b)** Loss scaling multiplies the gradient by $S$, adding $\log S$ to the budget:

$$
L^{*} = \frac{\log x_{\min} - \log S}{\mathbb{E}[\log\sigma]} = \frac{-16.63 - 11.09}{-0.1054} \approx 263 \text{ layers}
$$

Loss scaling buys $\log S / \vert\mathbb{E}[\log\sigma]\vert \approx 105$ layers — **depth, measured in layers**, which is the most concrete way to state what it purchases.

**(c)** Residual connections change the premise rather than the arithmetic. With $J_k = I + \tilde{J}_k$ and $\Vert\tilde{J}_k\Vert_2 = \beta$,

$$
\prod_k \sigma(J_k) \in \left[ (1-\beta)^{L}, (1+\beta)^{L} \right]
$$

and modern initializations arrange $\beta = O(1/\sqrt{L})$ (or $O(1/L)$ with zero-initialized branch outputs), so $\beta L = O(\sqrt{L})$ or $O(1)$ and the product stays $\Theta(1)$ **independently of depth**. Then $\mathbb{E}[\log\sigma] \approx 0$, $L^{*} = \infty$, and the underflow analysis becomes vacuous — which is why 100-layer residual networks train in fp16 while 30-layer plain networks do not.

$$
\boxed{L^{*} = \frac{\log x_{\min}}{\mathbb{E}[\log\sigma]}: \; 158 \; (\text{fp16}), \; 263 \; (\text{fp16} + S = 2^{16}), \; 980 \; (\text{fp32})}
$$

*Key takeaway*: depth, precision, and loss scale are one equation. Log the per-layer gradient norms, fit the slope, and this formula tells you which of the three to change.

### Problem L2.6: fp8 per-tensor scaling

An activation tensor has $\vert x \vert_{\max} = 0.05$ and $\vert x \vert_{\min} = 10^{-6}$ (nonzero entries), to be stored in fp8 E4M3 (max $448$, min normal $2^{-9} = 2.0\times10^{-3}$, $u = 2^{-4}$). (a) What happens without scaling? (b) Choose a power-of-two scale. (c) Check the small end. (d) Why is E5M2 used for gradients and E4M3 for activations?

**Solution**

**(a) Without scaling.** The largest value $0.05$ is $8960\times$ below the format maximum: the top four exponent bits are wasted. The smallest values $10^{-6}$ are **far below** the minimum normal $2\times10^{-3}$ and even below the subnormal floor, so they flush to zero. The tensor would be represented using a tiny sliver of the format's range — the classic reason naive fp8 casting destroys a model.

**(b) Scale.** Aim for $\vert x \vert_{\max}$ to land just under $448$:

$$
S = \frac{448}{0.05} = 8960 \quad \Longrightarrow \quad \text{round down to } S = 2^{13} = 8192
$$

Then $\max$ becomes $0.05 \times 8192 = 409.6 \lt 448$ ✓, with a small safety margin for the next step's amax drift (the amax-history mechanism keeps a running maximum over recent steps precisely for this).

**(c) Small end.** $10^{-6} \times 8192 = 8.2\times10^{-3} \gt 2.0\times10^{-3}$ ✓ — representable as a normal number. Feasibility in general requires

$$
\frac{\vert x \vert_{\max}}{\vert x \vert_{\min}} \lt \frac{448}{2.0\times10^{-3}} = 2.24\times10^{5} \approx 2^{17.8}
$$

Here the ratio is $0.05/10^{-6} = 5\times10^{4} \lt 2.24\times10^{5}$ ✓. A tensor whose dynamic range exceeds $\approx 18$ binades cannot fit E4M3 at any single scale — hence *per-channel* or *per-block* scaling for such tensors.

**(d)** E4M3 (4 exponent, 3 mantissa) has a narrower range but $2\times$ the precision of E5M2; E5M2 (5 exponent, 2 mantissa) matches fp16's range. Weights and activations have comparatively narrow, well-behaved dynamic range and benefit from precision; **gradients have a much wider dynamic range** (spanning many binades across layers, and shifting during training) and benefit from exponent bits. The assignment is the same range-versus-precision decision as bf16-versus-fp16, made per tensor role.

$$
\boxed{S = 2^{13}: \; \max \to 409.6 \lt 448, \; \min \to 8.2\times10^{-3} \gt 2.0\times10^{-3}; \text{ feasible while range} \lt 2^{17.8}}
$$

*Key takeaway*: per-tensor scaling is loss scaling generalized — one constant per tensor per step, chosen from the running amax, so each tensor fills the tiny fp8 window.

## Level 3 — Challenge

### Problem L3.1: Prove the log-sum-exp error bound

Prove that the max-subtracted algorithm for $L = \log\sum_{j=1}^{n}e^{z_j}$ commits absolute error $O((n + T)u)$, where $T = \max_j (m - z_j)$ is the logit spread, assuming `exp` and `log` are faithful to one rounding. Then explain why the naive algorithm admits no bound at all.

**Solution**

Write $m = \max_j z_j$, $t_j = z_j - m \in [-T, 0]$, $\Sigma = \sum_j e^{t_j} \in [1, n]$, so $L = m + \log\Sigma$.

**Step 1 — the subtraction.** $\mathrm{fl}(z_j - m) = t_j(1 + \delta_j)$ with $\vert\delta_j\vert \le u$. (For $j$ attaining the max this is exactly $0$.) Then

$$
e^{t_j(1+\delta_j)} = e^{t_j}e^{t_j\delta_j} = e^{t_j}\left( 1 + t_j\delta_j + O(u^{2}) \right)
$$

so the relative perturbation of term $j$ is at most $\vert t_j \vert u \le Tu$. **This is the only place the logit spread enters**, and it is why widely spread logits cost accuracy while uniformly shifted ones cost nothing.

**Step 2 — the exponential.** A faithful `exp` adds a further relative $u$: computed term $= e^{t_j}(1 + \eta_j)$, $\vert\eta_j\vert \le (T+1)u + O(u^{2})$.

**Step 3 — the sum.** All terms are **positive**, so the summation condition number $\frac{\sum\vert a_j\vert}{\vert\sum a_j\vert}$ equals exactly $1$: no cancellation is possible. Recursive summation (Topic 02, Theorem 2.3) gives

$$
\hat{\Sigma} = \Sigma(1 + \theta), \qquad \vert\theta\vert \le \gamma_{n-1} + \max_j\vert\eta_j\vert \le (n - 1)u + (T+1)u + O(u^{2})
$$

**Step 4 — the logarithm.** $\log$ turns relative input error into absolute output error:

$$
\log\hat{\Sigma} = \log\Sigma + \log(1+\theta) = \log\Sigma + \theta + O(\theta^{2})
$$

plus the library's own rounding, $\le u\vert\log\Sigma\vert \le u\log n$.

**Step 5 — the final addition.** Adding $m$ commits $\le u(\vert m \vert + \vert\log\Sigma\vert)$. Collecting:

$$
\left\vert \hat{L} - L \right\vert \; \le \; \underbrace{(n + T)u}_{\text{steps 1–3}} + \underbrace{u\log n}_{\text{step 4}} + \underbrace{u\vert L \vert}_{\text{step 5}} + O(u^{2}) \; = \; O\!\left( (n + T)u + u\vert L\vert \right) \qquad \blacksquare
$$

**Why the naive algorithm has no bound.** Its failure is not a large error constant but a *domain* failure: for $m \gt \log X_{\max}$ every term overflows and the computed value is $\infty$ — an error of $\infty$, admitting no bound of the form $C(n)u$. Symmetrically, for $m \lt \log X_{\min}$ all terms underflow to zero and the result is $-\infty$ or `NaN`. Between these limits the naive algorithm is accurate; outside them it is arbitrarily wrong. **An algorithm whose validity depends on the input's magnitude is not a numerical algorithm — max-subtraction removes the dependence entirely.**

$$
\boxed{\left\vert \hat{L} - L \right\vert \le (n + T)u + u\log n + u\vert L\vert + O(u^{2}); \; \text{naive: unbounded outside } [\log X_{\min}, \log X_{\max}]}
$$

*Key takeaway*: the bound depends on the logit *spread*, not their location — which is exactly why temperature scaling, $1/\sqrt{d_k}$, and logit capping are numerical interventions.

### Problem L3.2: A full error budget for one mixed-precision step

A layer computes $y = Wx$ with $H = 4096$, $W$ and $x$ in bf16, accumulation in fp32 with a tree reduction, in a data-parallel run with batch $B = 1024$. (a) Assemble the total relative error of $y$. (b) Compare with the gradient noise from mini-batch sampling. (c) Conclude whether bf16 is safe, and identify which term would break first if $H$ grew.

**Solution**

**(a) Error sources, in order.**

1. **Input rounding.** $\mathrm{fl}_{\text{bf}}(W) = W(1+\delta_W)$, $\mathrm{fl}_{\text{bf}}(x) = x(1+\delta_x)$ with $\vert\delta\vert \le u_{\text{bf}} = 3.9\times10^{-3}$. Each product therefore carries relative error $\le 2u_{\text{bf}} = 7.8\times10^{-3}$.
2. **Multiplication.** Tensor cores compute the products exactly in the wider accumulator format (bf16 $\times$ bf16 is exactly representable in fp32): contributes $0$.
3. **Accumulation.** Tree reduction of $H = 4096$ terms in fp32: $\gamma_{\log_2 H} \approx 12 \times 6.0\times10^{-8} = 7.2\times10^{-7}$.
4. **Output cast** back to bf16: another $u_{\text{bf}} = 3.9\times10^{-3}$.
5. **Cancellation.** If the dot product's terms have mixed signs, all of the above is amplified by $\kappa_{\text{sum}} = \frac{\sum\vert w_ix_i\vert}{\vert\sum w_ix_i\vert}$. For random-sign terms, $\kappa_{\text{sum}} \approx \sqrt{H} = 64$.

**Total** (well-conditioned case, $\kappa_{\text{sum}} = O(1)$):

$$
\frac{\vert\Delta y\vert}{\vert y \vert} \; \lesssim \; 2u_{\text{bf}} + \gamma_{\log H} + u_{\text{bf}} \approx 1.2\times10^{-2} \quad (\approx 1\%)
$$

With $\kappa_{\text{sum}} = 64$ the accumulation term rises to $4.6\times10^{-5}$ — still negligible; the input rounding dominates and is *not* amplified in the same way, since it perturbs the data rather than cancelling. Order of magnitude: **$1$–$5\%$ relative error on activations.**

**(b) Gradient noise.** A mini-batch gradient is a mean of $B$ i.i.d. per-example gradients, so its relative standard deviation is

$$
\frac{\mathrm{sd}}{\Vert g \Vert} \approx \frac{1}{\sqrt{B}} = \frac{1}{32} = 3.1\%
$$

**(c) Conclusion.** The arithmetic error ($\approx 1\%$) is **below the statistical noise the optimizer already tolerates** ($\approx 3\%$), and unlike the noise it is not systematically biased. That is the entire justification for low-precision training: SGD is a stochastic algorithm whose convergence depends on the noise *scale*, and bf16 arithmetic adds a perturbation smaller than the sampling noise already present.

**What breaks first as $H$ grows.** Not the accumulation ($\gamma_{\log H}$ grows only logarithmically in fp32, reaching $10^{-6}$ at $H = 10^{6}$), and not the input rounding (independent of $H$). The binding constraint is **cancellation**: $\kappa_{\text{sum}} \approx \sqrt{H}$ grows, and once $\kappa_{\text{sum}}u_{\text{bf}} \sim 1$ — around $H \sim u_{\text{bf}}^{-2} = 6.5\times10^{4}$ — the dot product loses all relative accuracy. This is why very wide layers, and especially attention over very long sequences, need fp32 accumulation *and* attention to the conditioning of the sum itself.

$$
\boxed{\text{bf16 + fp32 tree accumulate} \Rightarrow \approx 1\% \text{ error} \; \lt \; \tfrac{1}{\sqrt{B}} = 3\% \text{ sampling noise; cancellation binds first}}
$$

*Key takeaway*: low precision is safe because it is quieter than SGD's own noise. The correct comparison is never against fp64 — it is against $1/\sqrt{B}$.

### Problem L3.3: The softmax Jacobian, temperature, and gradient flow

For $p = \mathrm{softmax}(z/T)$: (a) derive the Jacobian; (b) bound its spectral norm; (c) analyze the limits $T \to 0$ and $T \to \infty$; (d) show the cross-entropy gradient is bounded and explain why the fused loss inherits this.

**Solution**

**(a)** With $p_i = e^{z_i/T}/\sum_j e^{z_j/T}$, differentiate:

$$
\frac{\partial p_i}{\partial z_k} = \frac{1}{T}\left( p_i\mathbb{1}[i=k] - p_ip_k \right) \quad \Longrightarrow \quad J = \frac{1}{T}\left( \mathrm{diag}(p) - pp^{\top} \right)
$$

**(b)** $M = \mathrm{diag}(p) - pp^{\top}$ is symmetric positive semidefinite: for any $v$,

$$
v^{\top}Mv = \sum_i p_iv_i^{2} - \left( \sum_i p_iv_i \right)^{2} = \mathrm{Var}_{i \sim p}(v_i) \ge 0
$$

— the Jacobian **is the covariance operator of the distribution $p$**. It annihilates the constant vector ($M\mathbf{1} = 0$), reflecting softmax's shift invariance, so $J$ is always singular: its smallest eigenvalue is exactly $0$ and any "condition number" must be taken on the complement of $\mathbf{1}$. Bounding the top:

$$
\Vert M \Vert_2 \le \max_i \mathrm{Var}(v) \text{ over unit } v \le \max_i p_i(1-p_i) \cdot \text{(const)} \le \tfrac{1}{4}\cdot\text{(const)}, \qquad \mathrm{tr}(M) = 1 - \Vert p \Vert_2^{2} \le 1
$$

so $\Vert J \Vert_2 \le 1/T$ and in fact $\Vert J \Vert_2 \le \frac{1}{T}\max_i p_i \le \frac{1}{T}$.

**(c) Limits.**

- $T \to 0$: $p \to$ one-hot, $\Vert p \Vert_2 \to 1$, $\mathrm{tr}(M) \to 0$ — the Jacobian **vanishes**: a saturated softmax passes no gradient. The $1/T$ prefactor grows, but the exponential collapse of $M$ wins. This is the numerical face of "confident predictions have no gradient", and the reason label smoothing, logit capping, and temperature help.
- $T \to \infty$: $p \to$ uniform $(1/n)$, $M \to \frac{1}{n}(I - \frac{1}{n}\mathbf{1}\mathbf{1}^{\top})$, $\Vert J \Vert_2 \to \frac{1}{nT} \to 0$ — the Jacobian also vanishes, now because the output is insensitive to any input.

Gradient flow is maximized at intermediate $T$, i.e. logits of order $1$ — precisely the regime the $1/\sqrt{d_k}$ scaling of Problem L2.1 enforces, and precisely the regime where the exponentials are safest.

**(d) Cross-entropy gradient.** For $\ell = -\log p_y$ (at $T = 1$), the chain rule collapses:

$$
\frac{\partial\ell}{\partial z_i} = \sum_k \frac{\partial\ell}{\partial p_k}\frac{\partial p_k}{\partial z_i} = -\frac{1}{p_y}\cdot\left( p_y\mathbb{1}[i=y] - p_yp_i \right) = p_i - \mathbb{1}[i=y] \in [-1, 1]
$$

The dangerous factor $1/p_y$ — which overflows when $p_y$ underflows — **cancels analytically against the $p_y$ in the Jacobian**. A fused kernel performs this cancellation symbolically and never forms $1/p_y$; an unfused pipeline computes both factors numerically, and in fp16 the first is `inf` while the second is $0$, giving `NaN`.

$$
\boxed{J = \tfrac{1}{T}\!\left( \mathrm{diag}(p) - pp^{\top} \right) = \tfrac{1}{T}\mathrm{Cov}(p), \; \Vert J \Vert_2 \le \tfrac{1}{T}; \quad \tfrac{\partial\ell}{\partial z} = p - e_y \in [-1,1]}
$$

*Key takeaway*: the softmax Jacobian is a covariance, so it vanishes at both temperature extremes; and the fused cross-entropy is stable because a symbolic cancellation removes the only unbounded factor.

### Problem L3.4: Stochastic rounding rescues absorbed updates

Round-to-nearest discards any update below $\tfrac{1}{2}\mathrm{ulp}$. **Stochastic rounding** rounds $x$ to a neighbouring grid point $x_{\downarrow} \le x \le x_{\uparrow}$ with probability proportional to proximity:

$$
\Pr[x \to x_{\uparrow}] = \frac{x - x_{\downarrow}}{x_{\uparrow} - x_{\downarrow}}, \qquad \Pr[x \to x_{\downarrow}] = \frac{x_{\uparrow} - x}{x_{\uparrow} - x_{\downarrow}}
$$

(a) Prove it is unbiased. (b) Show that $n$ absorbed updates accumulate correctly in expectation, with variance you should quantify. (c) Compare with the fp32 master-weight approach. (d) State the caveat.

**Solution**

**(a) Unbiasedness.** Let $\Delta = x_{\uparrow} - x_{\downarrow} = \mathrm{ulp}$. Then

$$
\mathbb{E}[\mathrm{SR}(x)] = x_{\uparrow}\frac{x - x_{\downarrow}}{\Delta} + x_{\downarrow}\frac{x_{\uparrow} - x}{\Delta} = \frac{x_{\uparrow}x - x_{\uparrow}x_{\downarrow} + x_{\downarrow}x_{\uparrow} - x_{\downarrow}x}{\Delta} = \frac{x(x_{\uparrow} - x_{\downarrow})}{\Delta} = x \qquad \blacksquare
$$

so $\mathrm{SR}(x) = x + \xi$ with $\mathbb{E}[\xi] = 0$ and $\vert\xi\vert \lt \Delta$. Round-to-nearest, by contrast, is **biased whenever the update is small**: for $\vert\delta\vert \lt \Delta/2$ it returns $x$ with probability $1$, i.e. bias $= -\delta$ — the entire update.

**(b) Accumulation.** Apply $n$ updates $\theta \leftarrow \mathrm{SR}(\theta + \delta)$ with a small constant $\delta$ and $\delta \ll \Delta$. Under round-to-nearest, $\theta$ never changes: total drift $0$ instead of $n\delta$. Under stochastic rounding each step moves $\theta$ up by $\Delta$ with probability $\delta/\Delta$ and stays otherwise, so after $n$ steps the displacement $D_n$ satisfies

$$
\mathbb{E}[D_n] = n\Delta\cdot\frac{\delta}{\Delta} = n\delta \quad \checkmark \text{ (exactly the intended drift)}
$$

$$
\mathrm{Var}(D_n) = n\Delta^{2}\frac{\delta}{\Delta}\left(1 - \frac{\delta}{\Delta}\right) \approx n\Delta\delta \quad \Longrightarrow \quad \mathrm{sd}(D_n) \approx \sqrt{n\Delta\delta}
$$

The **relative** noise is $\frac{\sqrt{n\Delta\delta}}{n\delta} = \sqrt{\frac{\Delta}{n\delta}} \to 0$: the signal grows like $n$ while the noise grows like $\sqrt{n}$, so the accumulated trajectory is correct to relative accuracy $\sqrt{\Delta/(n\delta)}$ after $n$ steps. **A biased $O(1)$ error has been converted into an unbiased random walk.**

**(c) Versus master weights.** Both solve absorption, differently:

| | fp32 master weights | Stochastic rounding |
|---|---|---|
| Memory | $+4$ bytes/param | none |
| Determinism | exact reproducibility | needs seeded RNG per element |
| Hardware | universal | needs SR support in the FMA/cast unit |
| Error | deterministic, no drift | unbiased, $\sqrt{n}$ noise |

Stochastic rounding is the memory-efficient option, and is why bf16-only training (no master copy) becomes viable on hardware that supports it — a $4$ bytes/parameter saving, i.e. $28$ GB on a 7B model.

**(d) Caveat.** Unbiasedness holds *per rounding*, not for nonlinear compositions: $\mathbb{E}[f(\mathrm{SR}(x))] \ne f(x)$ in general, so SR does not make an entire training step unbiased. It also injects extra gradient noise, which interacts with the learning-rate schedule, and it costs an RNG stream per element. As always, the correct comparison is against the noise already present — Problem L3.2's $1/\sqrt{B}$.

$$
\boxed{\mathbb{E}[\mathrm{SR}(x)] = x; \; \mathbb{E}[D_n] = n\delta \text{ with } \mathrm{sd} \approx \sqrt{n\Delta\delta}: \text{ relative noise } \sqrt{\Delta/(n\delta)} \to 0}
$$

*Key takeaway*: rounding *bias* is the enemy, not rounding *error*. Any device that converts a systematic loss into a zero-mean fluctuation — stochastic rounding, Kahan compensation, an fp32 master copy — buys back the accumulation.